# 04.1 Regresión lineal y logística (experimento 1)

- Entrada: `data/processed/dev_features.csv` (salida del notebook 03)
- Salida: resultados de validación cruzada de este modelo, para compararlos con los demás

Instrucciones:

- Validación: `GroupKFold(n_splits=5, shuffle=True, random_state=SEED)` agrupando por `animal_id_intake`. Es la misma en todos los notebooks.
- Hiperparámetros: salen de la búsqueda con Optuna del mismo modelo (carpeta `04-hyperparameter-optimization-experiment-1`), con `load_best(...)` y los nombres `linear_clf` y `linear_reg`. Si las variables o los datos cambian, hay que repetir esa búsqueda.
- Preprocesamiento: `build_preprocessor(columns=...)` dentro de un `Pipeline`, para que se ajuste en cada fold.
- Métricas: clasificación con macro-F1 y reporte por clase. Regresión con MAE en días (`np.expm1`) y R² en log.
- Resultados por fold: mostrar también el resultado de cada fold, como en `04.1-svm.ipynb` (sección 4).
- No usar `test`.

In [ ]:
# 04.1 Regresión lineal y logística (experimento 1)

import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score, mean_absolute_error, r2_score

from config import SEED
from features import CAT_LOW, CAT_HIGH, BINARY, NUMERIC, FEATURES, build_preprocessor
from tuning import load_best

data = pd.read_csv("../../data/processed/dev_features.csv")
groups = data["animal_id_intake"]
cv = GroupKFold(n_splits=5, shuffle=True, random_state=SEED)

## Inputs

- `AVAILABLE`: las 22 variables de `dev_features.csv`.
- `FEATURES_EXP1_CLF` y `FEATURES_EXP1_REG`: variables del experimento 1 para este modelo, en clasificación y en regresión. Con regularización admite todas las variables.

In [ ]:
AVAILABLE = FEATURES
FEATURES_EXP1_CLF = AVAILABLE
FEATURES_EXP1_REG = AVAILABLE

print("El experimento 1 usa estas variables:")
print("Clasificación:", FEATURES_EXP1_CLF)
print("Regresión:", FEATURES_EXP1_REG)

# Clasificación: todas las filas
y_clf = data["y_outcome"]

# Regresión: solo adoptados
adopted = data["y_outcome"] == "Adoption"
y_reg = data.loc[adopted, "y_log_days"]
groups_reg = groups[adopted]

El experimento 1 usa estas variables:
Clasificación: ['animal_type', 'intake_type', 'intake_condition', 'sex', 'weekday', 'season', 'breed_main', 'color_main', 'city', 'is_neutered', 'is_mix', 'is_crossbreed', 'is_bicolor', 'is_tabby', 'has_location_detail', 'is_baby', 'condition_serious', 'age_days', 'prior_stays', 'hour', 'month', 't_months']
Regresión: ['animal_type', 'intake_type', 'intake_condition', 'sex', 'weekday', 'season', 'breed_main', 'color_main', 'city', 'is_neutered', 'is_mix', 'is_crossbreed', 'is_bicolor', 'is_tabby', 'has_location_detail', 'is_baby', 'condition_serious', 'age_days', 'prior_stays', 'hour', 'month', 't_months']


## Hiperparámetros

Salen de la búsqueda con Optuna del experimento 1. Si las variables o los datos cambiaron desde esa búsqueda, `load_best` se detiene y avisa que hay que repetirla.

In [ ]:
params_clf = load_best("linear_clf", FEATURES_EXP1_CLF)
params_reg = load_best("linear_reg", FEATURES_EXP1_REG)

## 1. Clasificación

Objetivo: `y_outcome` (5 clases), con todas las filas.

In [ ]:
# --- 1. Clasificación ---
# LogisticRegression con max_iter=2000 para evitar UserWarning por límite de iteraciones
pipe_clf = Pipeline([("prep", build_preprocessor(columns=FEATURES_EXP1_CLF)), 
                     ("model", LogisticRegression(max_iter=2000, **params_clf))])
X_clf = data[FEATURES_EXP1_CLF]

pred_clf = cross_val_predict(pipe_clf, X_clf, y_clf, groups=groups, cv=cv, n_jobs=5)
base_clf = cross_val_predict(DummyClassifier(strategy="most_frequent"), X_clf, y_clf, groups=groups, cv=cv)

print(f"macro-F1: {f1_score(y_clf, pred_clf, average='macro'):.3f} (base: {f1_score(y_clf, base_clf, average='macro'):.3f})\n")
print(classification_report(y_clf, pred_clf))

ConfusionMatrixDisplay.from_predictions(y_clf, pred_clf, normalize="true", xticks_rotation=30, values_format=".2f")
plt.show()

## 2. Regresión

Objetivo: `y_log_days`, solo con las filas donde `y_outcome == "Adoption"`.

In [ ]:
# --- 2. Regresión ---
pipe_reg = Pipeline([("prep", build_preprocessor(columns=FEATURES_EXP1_REG)), 
                     ("model", Ridge(**params_reg))])
X_reg = data.loc[adopted, FEATURES_EXP1_REG]

pred_log = cross_val_predict(pipe_reg, X_reg, y_reg, groups=groups_reg, cv=cv, n_jobs=5)
base_log = cross_val_predict(DummyRegressor(strategy="median"), X_reg, y_reg, groups=groups_reg, cv=cv)

days_real = np.expm1(y_reg)
pd.DataFrame({
    "R² (log)": [r2_score(y_reg, pred_log), r2_score(y_reg, base_log)],
    "MAE (días)": [mean_absolute_error(days_real, np.expm1(pred_log)), mean_absolute_error(days_real, np.expm1(base_log))],
}, index=["Ridge", "Siempre la mediana"]).round(3)

## 3. Resultados

In [16]:
# --- 3. Resultados Globales ---
print("\n--- Resultados Resumidos ---")
pd.DataFrame({
    "Clasificación (macro-F1)": [round(f1_score(y_clf, pred_clf, average="macro"), 3)],
    "Regresión (R² log)": [round(r2_score(y_reg, pred_log), 3)],
    "Regresión (MAE días)": [round(mean_absolute_error(days_real, np.expm1(pred_log)), 3)],
}, index=["Linear/Logistic"])


--- Resultados Resumidos ---


,Clasificación (macro-F1),Regresión (R² log),Regresión (MAE días)
Linear/Logistic,0.481,0.222,22.229


## 4. Resultados por fold

Cada fold deja fuera un grupo distinto de animales para evaluar. Estos son los resultados de cada uno, con los mismos folds en todos los modelos, así que se pueden comparar fold por fold. No vuelve a entrenar nada: usa las predicciones de arriba.

In [10]:
# --- 4. Resultados por fold ---
rows_clf = []
for fold, (_, val) in enumerate(cv.split(X_clf, y_clf, groups), start=1):
    rows_clf.append({"fold": fold, "filas": len(val),
                     "macro-F1": f1_score(y_clf.iloc[val], pred_clf[val], average="macro"),
                     "exactitud": (pred_clf[val] == y_clf.iloc[val]).mean()})

rows_reg = []
for fold, (_, val) in enumerate(cv.split(X_reg, y_reg, groups_reg), start=1):
    rows_reg.append({"fold": fold, "filas": len(val),
                     "R² (log)": r2_score(y_reg.iloc[val], pred_log[val]),
                     "MAE (días)": mean_absolute_error(days_real.iloc[val], np.expm1(pred_log[val]))})

for title, rows in [("Clasificación", rows_clf), ("Regresión", rows_reg)]:
    table = pd.DataFrame(rows).set_index("fold")
    summary = table.drop(columns="filas").agg(["mean", "std"])
    print(f"\n=== {title}: resultado por fold ===")
    print(table.round(3).to_string())
    print(summary.round(3).to_string(), "\n")


=== Clasificación: resultado por fold ===
      filas  macro-F1  exactitud
fold                            
1     12729     0.477      0.605
2     12783     0.481      0.605
3     12697     0.478      0.600
4     12733     0.482      0.606
5     12739     0.488      0.610
      macro-F1  exactitud
mean     0.481      0.605
std      0.004      0.003 


=== Regresión: resultado por fold ===
      filas  R² (log)  MAE (días)
fold                             
1      5393     0.215      22.548
2      5453     0.195      22.478
3      5401     0.248      22.240
4      5405     0.214      22.108
5      5421     0.235      21.770
      R² (log)  MAE (días)
mean     0.221      22.229
std      0.021       0.312 

